<a href="https://colab.research.google.com/github/hitesh771/Building-Segmentation-Detectron2/blob/main/2_Inference_App.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# === IMPORTS ===
# Run this cell once to install Detectron2 and import everything
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'

import detectron2
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2 import model_zoo
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog, DatasetCatalog
from detectron2.data.datasets import register_coco_instances # <-- This is the fix
import cv2
import os
import numpy as np
from google.colab.patches import cv2_imshow
from google.colab import files # For upload and download
import shutil # For zipping
from google.colab import drive

# === 1. CONNECT TO GOOGLE DRIVE ===
print("Connecting to Google Drive...")
drive.mount('/content/drive')

# === 2. SETUP THE MODEL CONFIGURATION ===
print("\nSetting up model configuration...")
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"))
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1  # 1 class ("building")
cfg.MODEL.DEVICE = "cuda" # Use the GPU

# === 3. LOAD YOUR PERMANENT, TRAINED MODEL ===
print("Loading your trained model from Google Drive...")
model_path = "/content/drive/MyDrive/my_building_model.pth"
cfg.MODEL.WEIGHTS = model_path
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.7  # Confidence threshold
predictor = DefaultPredictor(cfg)

# (Register metadata for visualization)
if "whu_buildings_train" not in DatasetCatalog.list():
    register_coco_instances("whu_buildings_train", {},
                            "/content/drive/MyDrive/WHOdataset/WHU/train/_annotations.json",
                            "/content/drive/MyDrive/WHOdataset/WHU/train/Image")
building_metadata = MetadataCatalog.get("whu_buildings_train")

print("✅ Your custom building-detection model is loaded and ready!")

# === 4. GET THE INPUT IMAGE (User Upload) ===
print("\n--- Please Upload Your Aerial Image ---")
uploaded = files.upload()

if len(uploaded) == 0:
    print("No file uploaded. Please run the cell again.")
else:
    # Get the filename and file content
    filename = list(uploaded.keys())[0]
    file_bytes = uploaded[filename]

    print(f"\nProcessing uploaded file: {filename}")

    # Convert the bytes to an image
    nparr = np.frombuffer(file_bytes, np.uint8)
    im = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

    # --- 5. RUN PREDICTION ---
    print(f"Running prediction on {filename}...")
    outputs = predictor(im)

    # --- 6. DISPLAY THE FULL SEGMENTED IMAGE ---
    print("Displaying full segmentation result:")
    v = Visualizer(im[:, :, ::-1],
                   metadata=building_metadata,
                   scale=0.8
    )
    out = v.draw_instance_predictions(outputs["instances"].to("cpu"))
    cv2_imshow(out.get_image()[:, :, ::-1])

    # --- 7. STORE EXTRACTED FILES ---
    print("\nExtracting all found buildings...")
    instances = outputs["instances"].to("cpu")
    boxes = instances.pred_boxes.tensor.numpy()
    masks = instances.pred_masks.numpy()

    output_dir_buildings = "extracted_buildings"
    if os.path.exists(output_dir_buildings):
        shutil.rmtree(output_dir_buildings)
    os.makedirs(output_dir_buildings, exist_ok=True)

    for i in range(len(boxes)):
        x1, y1, x2, y2 = boxes[i].astype(int)
        mask = masks[i]

        # Create a 4-channel transparent (BGRA) image
        masked_crop = np.zeros((y2-y1, x2-x1, 4), dtype=np.uint8)
        masked_crop[:,:,:3] = im[y1:y2, x1:x2]
        masked_crop[:,:, 3] = mask[y1:y2, x1:x2] * 255

        output_filename = os.path.join(output_dir_buildings, f"building_{i+1}.png")
        cv2.imwrite(output_filename, masked_crop)

    print(f"✅ Success! {len(boxes)} buildings were saved to the '{output_dir_buildings}' folder.")

    # --- 8. PROVIDE DOWNLOAD OPTION ---
    print("\nZipping files for download...")
    shutil.make_archive("building_results", 'zip', output_dir_buildings)

    print("✅ Zip file created. Triggering download...")
    files.download('building_results.zip')